In [1]:
import random

# Define possible nucleotides
nucleotides = ['A', 'C', 'G', 'T']

# Define mutation rate per site (chance that a site will mutate each generation)
mutation_rate = 0.1

# Define the number of generations to simulate
num_generations = 10

# Initialize a random starting sequence (finite length)
sequence_length = 20
sequence = [random.choice(nucleotides) for _ in range(sequence_length)]

def mutate_site(nucleotide):
    """Mutate a given nucleotide to one of the other three nucleotides."""
    possible_mutations = [nuc for nuc in nucleotides if nuc != nucleotide]
    return random.choice(possible_mutations)

# Simulate sequence evolution over generations
print("Generation 0:", ''.join(sequence))

for generation in range(1, num_generations + 1):
    new_sequence = []
    for site in sequence:
        # Mutate with a probability equal to mutation_rate
        if random.random() < mutation_rate:
            new_sequence.append(mutate_site(site))
        else:
            new_sequence.append(site)
    sequence = new_sequence  # Update sequence for the next generation
    print(f"Generation {generation}: {''.join(sequence)}")


Generation 0: ACATATGCTGTTGGTTACTC
Generation 1: ACATATGCTGTTGGTTACGC
Generation 2: ACATACGCTGTTGGTTAGGC
Generation 3: ACGTACGCTGTTTGTTAGGC
Generation 4: ACGTACGATGTTTGTGATGC
Generation 5: ACGTACGATGTTTGTTATGC
Generation 6: ACGTACGATGTATGTTATGC
Generation 7: ACGTACGATGTATGTTATGC
Generation 8: CCGTACGATGTATGTTATGA
Generation 9: GCGTTCGTTGTATGTTATGA
Generation 10: GCGGTCGTTGTATATTTTGA


In [14]:
import numpy as np

# Dummy functions for calculating likelihood, generating trees, and models
def calculate_likelihood(tree, theta, model):
    # Placeholder likelihood function
    # Acknowledge the parameters to avoid compile errors
    _ = tree
    _ = theta
    _ = model
    return np.random.uniform(0, 1)  # returns a random likelihood for demonstration

def generate_random_tree(num_leaves):
    # Placeholder for a random tree structure generation
    return {"tree_structure": f"Tree with {num_leaves} leaves"}

def sample_theta():
    # Placeholder for sampling new parameter theta
    return np.random.uniform(0, 1)

def sample_model():
    # Placeholder for sampling new model configuration
    return {"model_param": np.random.uniform(0, 1)}

def phylo_tree_search(D, theta_0, M_lambda_0, n_iter, p1, p2, m, mh):
    # Initialize
    T = generate_random_tree(m)  # Random initial tree with m leaves
    theta = theta_0
    M_lambda = M_lambda_0
    L = calculate_likelihood(T, theta, M_lambda)

    # Track best results
    L_best = L
    T_best = T
    theta_best = theta
    M_lambda_best = M_lambda

    # Iterative search
    for i in range(n_iter):
        # Propose new theta and model
        if np.random.uniform(0, 1) <= p1:
            theta_new = sample_theta()
            M_lambda_new = M_lambda
            rho_theta = min(1, calculate_likelihood(T, theta_new, M_lambda) / L)
            if np.random.uniform(0, 1) <= rho_theta:
                theta = theta_new
                M_lambda = M_lambda_new
        elif np.random.uniform(0, 1) <= (p1 + p2):
            M_lambda_new = sample_model()
            theta_new = theta
            rho_lambda = min(1, calculate_likelihood(T, theta, M_lambda_new) / L)
            if np.random.uniform(0, 1) <= rho_lambda:
                M_lambda = M_lambda_new
        else:
            # Propose a new tree
            T_new = generate_random_tree(m)
            r_T = min(1, calculate_likelihood(T_new, theta, M_lambda) / L)
            if np.random.uniform(0, 1) <= r_T:
                T = T_new

        # Calculate likelihood with updated values
        L = calculate_likelihood(T, theta, M_lambda)

        # Update best results if likelihood improves
        if L > L_best:
            L_best = L
            T_best = T
            theta_best = theta
            M_lambda_best = M_lambda

    # Return the best tree and parameters
    return L_best, T_best, theta_best, M_lambda_best

# Parameters for the search
D = None  # Placeholder for data
theta_0 = 0.5  # Initial theta
M_lambda_0 = {"model_param": 0.5}  # Initial model parameters
n_iter = 100  # Number of iterations
p1, p2 = 0.3, 0.3  # Probabilities for parameter/model updates
m = 5  # Number of leaves in the tree
mh = 0.1  # Metropolis-Hastings threshold (for simplicity, not directly used here)

# Run the phylogenetic tree search
best_likelihood, best_tree, best_theta, best_model = phylo_tree_search(D, theta_0, M_lambda_0, n_iter, p1, p2, m, mh)

print("Best Likelihood:", best_likelihood)
print("Best Tree:", best_tree)
print("Best Theta:", best_theta)
print("Best Model:", best_model)


Best Likelihood: 0.9951065965093829
Best Tree: {'tree_structure': 'Tree with 5 leaves'}
Best Theta: 0.6541003228041742
Best Model: {'model_param': 0.5102863840389731}


In [ ]:
'''
chr1	1	248956422
chr2	1	242193529
chr3	1	198295559
chr4	1	190214555
chr5	1	181538259
chr6	1	170805979
chr7	1	159345973
chr8	1	145138636
chr9	1	138394717
chr10	1	133797422
chr11	1	135086622
chr12	1	133275309
chr13	1	114364328
chr14	1	107043718
chr15	1	101991189
chr16	1	90338345
chr17	1	83257441
chr18	1	80373285
chr19	1	58617616
chr20	1	64444167
chr21	1	46709983
chr22	1	50818468
chrX	1	156040895
chrY	1	57227415
chrM	1	16569
'''

In [34]:
import random
def batch_config_window(configItem, groupNum = None):
    """
    Splits a configuration item into a list of windows based on the specified group number.
    Args:
        configItem (str): A string in the format "chr_start_end" where 'chr' is the chromosome,
                            'start' is the starting position, and 'end' is the ending position.
        groupNum (int, optional): The number of groups to split the window into. If not provided,
                                    defaults to 5.
    Returns:
        list: A list of lists, where each sublist contains the chromosome, start, and end positions
                of each window.
    Example:
        >>> batch_config_window("chr1_0_1000", 5)
        [['chr1', 1, 200], ['chr1', 200, 400], ['chr1', 400, 600], ['chr1', 600, 800], ['chr1', 800, 1000]]
    """
    res = []
    chr = configItem.split("_")[0]
    end = int(configItem.split("_")[2])
    # Set default groupNum if not provided
    if groupNum is None:
        groupNum = 5
    # Determine window size based on groupNum
    winSize = int(end / groupNum)
    # Create list of window start points, ensuring the last point matches 'end'
    configWindowList = list(range(1, end, winSize)) + [end]
    # Generate windows based on configWindowList
    for i in range(1, len(configWindowList)):
        restmp = [chr, configWindowList[i-1], configWindowList[i]]
        res.append(restmp)
    return res

k = batch_config_window("chr1_1_248956422", groupNum = 2876)
# randomly select 10 windows
random.sample(k, 10)

[['chr1', 67432578, 67519141],
 ['chr1', 56525640, 56612203],
 ['chr1', 121707579, 121794142],
 ['chr1', 229651640, 229738203],
 ['chr1', 138760490, 138847053],
 ['chr1', 224284734, 224371297],
 ['chr1', 35317705, 35404268],
 ['chr1', 179012285, 179098848],
 ['chr1', 176242269, 176328832],
 ['chr1', 226708498, 226795061]]

In [20]:
133133895 - 133047332

86563

In [337]:
import scipy.stats as stats
altReadsnum = 2
totalReadsnum = 100
lowBaseCutoff = 20
print(10 ** -(lowBaseCutoff/10))
prob = 1 - stats.binom.cdf(altReadsnum, totalReadsnum, 10 ** -(lowBaseCutoff/10))
print(prob)
if prob <= 0.0005:
    print("significant")

0.01
0.07937320225218047


In [168]:
data = open("/Users/liqun/scMutraceProject/Data/GSE227690/data/random/random_vcf_1.vcf", "r")
for line in data:
    if line.startswith("#"):
        continue
    line = line.strip().split("\t")
    print(line)
    totalReadsnum = int(line[3])
    altReadsnum = int(line[5])
    print(totalReadsnum)
    print(altReadsnum)
    break

['1', '12589539', 'C', '13', '0:0:13:0:0:0', '0', '10', '0:0:10:0:0:0', '0', 'GATAGAAAGCTCGACC;TTACTGTAGCCTGACC;GACCCAGGTCCTGTTC;TCAGTGACAAGGCAAC;TCTTAGTCATCGGTTA;TCCGGGAAGATTTGCC;TGGGATTGTGGCTACC;CAACAACTCATTTCGT;TCGCACTGTGTACATC;ATGGGAGGTTGAGGAC', '0|0|1|0|0|0;0|0|1|0|0|0;0|0|1|0|0|0;0|0|3|0|0|0;0|0|1|0|0|0;0|0|1|0|0|0;0|0|2|0|0|0;0|0|1|0|0|0;0|0|1|0|0|0;0|0|1|0|0|0', '||37|||;||37|||;||37|||;||37:37:37|||;||37|||;||37|||;||37:37|||;||37|||;||37|||;||37|||', 'None', 'None', '0;1;2;3;4;5;6;7;8;9', '0|0|1|0|0|0;0|0|1|0|0|0;0|0|1|0|0|0;0|0|3|0|0|0;0|0|1|0|0|0;0|0|1|0|0|0;0|0|2|0|0|0;0|0|1|0|0|0;0|0|1|0|0|0;0|0|1|0|0|0']
13
0


In [ ]:
import scipy.stats as stats
def generate_random_pool(targetVcfPath, pos):
    data = open(targetVcfPath, "r")
    totalReadsnum = 0
    altReadsnum = 0
    for line in data:
        if line.startswith("#"):
            continue
        line = line.strip().split("\t")
        if int(line[1]) == pos:
            continue
        totalReadsnum = totalReadsnum + int(line[3])
        altReadsnum = altReadsnum + int(line[5])
    randomPool = [0] * (totalReadsnum - altReadsnum) + [1] * altReadsnum
    return randomPool

def random_sample(randomPool, sampleNum, lowBaseCutoff):
    tmpRandom = random.sample(randomPool, sampleNum)
    prob = 1 - stats.binom.cdf(sum(tmpRandom), sampleNum, 10 ** -(lowBaseCutoff/10))
    if prob <= 0.0005:
        return True

randomPool = generate_random_pool("/Users/liqun/scMutraceProject/Data/GSE227690/data/random/random_vcf_1.vcf", 28370179)
print(len(randomPool), sum(randomPool))
tmpC = 0
for i in range(1000):
    if random_sample(randomPool, 16, 20):
        tmpC = tmpC + 1
print(tmpC)



2990 15
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0,

In [28]:
import scipy.io
mat = scipy.io.loadmat('/Users/liqun/Downloads/Data_Figure1_and_Figure2.mat')
print(mat.keys())

dict_keys(['__header__', '__version__', '__globals__', 'Data_Figure1_and_Figure2'])


In [31]:
mat.values()


dict_values([b'MATLAB 5.0 MAT-file, Platform: MACI64, Created on: Thu Sep 10 21:40:47 2020', '1.0', [], array([[(<14713x16360 sparse matrix of type '<class 'numpy.float64'>'
        	with 39465483 stored elements in Compressed Sparse Column format>, array([[3],
               [2],
               [4],
               ...,
               [5],
               [2],
               [5]], dtype=uint8), array([[ 17896.52 , -15445.141],
               [-14132.653, -12096.23 ],
               [ -5733.668,  -3052.766],
               ...,
               [  8690.236,   9175.516],
               [-12052.939,  -9058.457],
               [ 17484.047,  16522.252]]), array([[array(['BAS-I'], dtype='<U5'), array(['BAS-II'], dtype='<U6'),
                array(['BAS-III'], dtype='<U7'), array(['BAS-IV'], dtype='<U6'),
                array(['SPN'], dtype='<U3'), array(['GRN'], dtype='<U3'),
                array(['MEL'], dtype='<U3')]], dtype=object), array([[7]], dtype=uint8), array([[0.      , 0.447   , 

In [29]:
# convert to pandas dataframe
import pandas as pd
import numpy as np

Data_Figure1_and_Figure2 = pd.DataFrame({name: col.ravel() for name, col in mat.items() if isinstance(col, np.ndarray) and np.issubdtype(col.dtype, np.number)})



In [30]:
Data_Figure1_and_Figure2

""
